# Dataset Recheck :

## 1. File Inventory

### Boyne Resorts : Loon , Sunday River , Sugarloaf

#### Loon 

In [1]:
import json
from pathlib import Path
import pandas as pd
from pprint import pprint

In [2]:
loon_fp = Path("data/raw/loon/Loon_reportpal_20260303_0022.json")

with open(loon_fp, "r") as f:
    loon_payload = json.load(f)

In [3]:
print(type(loon_payload))
print(loon_payload.keys())

<class 'dict'>
dict_keys(['name', 'units', 'updated', 'liftsUpdated', 'trailsUpdated', 'commentsUpdated', 'operations', 'currentConditions', 'comments', 'facilities', 'summer', 'pricing', 'icons'])


In [4]:
for k, v in loon_payload.items():
    print(f"\n{k}: {type(v)}")


name: <class 'str'>

units: <class 'str'>

updated: <class 'str'>

liftsUpdated: <class 'str'>

trailsUpdated: <class 'str'>

commentsUpdated: <class 'str'>

operations: <class 'dict'>

currentConditions: <class 'dict'>

comments: <class 'dict'>

facilities: <class 'dict'>

summer: <class 'dict'>

pricing: <class 'NoneType'>

icons: <class 'dict'>


In [5]:
facilities = loon_payload.get("currentConditions", {}) or {}
print("facilities keys:")
print(facilities.keys())

areas = facilities.get("areas", {}) or {}
print("\nareas keys:")
print(areas.keys())

area_list = areas.get("area", []) or []
print("\nnumber of areas:", len(area_list))

if area_list:
    print("\nfirst area keys:")
    print(area_list[0].keys())

resortwide = facilities.get("resortwide", {}) or {}
print("\nresortwide keys:")
print(resortwide.keys())

trails_total = resortwide.get("numTrailsTotal")
trails_open = resortwide.get("numTrailsOpen")

lifts_total = resortwide.get("numLiftsTotal")
lifts_open = resortwide.get("numLiftsOpen")

comments = loon_payload.get("comments", {}) or {}
comment_list = comments.get("comment", []) or []

trail_report = None
if comment_list and isinstance(comment_list[0], dict):
    trail_report = comment_list[0].get("text")

print("trails_total:", trails_total)
print("trails_open:", trails_open)
print("lifts_total:", lifts_total)
print("lifts_open:", lifts_open)
print("trail_report:", trail_report[:300] if isinstance(trail_report, str) else trail_report)

assert trails_total is not None, "Missing numTrailsTotal"
assert trails_open is not None, "Missing numTrailsOpen"
assert lifts_total is not None, "Missing numLiftsTotal"
assert lifts_open is not None, "Missing numLiftsOpen"

assert trails_total >= 0
assert trails_open >= 0
assert lifts_total >= 0
assert lifts_open >= 0

assert trails_open <= trails_total, "numTrailsOpen > numTrailsTotal"
assert lifts_open <= lifts_total, "numLiftsOpen > numLiftsTotal"

open_trails_pct = int(round((trails_open / trails_total) * 100)) if trails_total else None
open_lifts_pct = int(round((lifts_open / lifts_total) * 100)) if lifts_total else None

print("open_trails_pct:", open_trails_pct)
print("open_lifts_pct:", open_lifts_pct)

facilities keys:
dict_keys(['resortwide', 'resortLocations', 'roads'])

areas keys:
dict_keys([])

number of areas: 0

resortwide keys:
dict_keys(['totalTrailArea', 'totalTrailLength', 'areaOpen', 'lengthOpen', 'numTrailsTotal', 'numTrailsOpen', 'numTrailsGroomed', 'numTrailsSnowMaking', 'numLiftsTotal', 'numLiftsOpen', 'numLiftsOnHold', 'numLiftsClosed', 'numParksTotal', 'numParksOpen', 'numPipesTotal', 'numPipesOpen', 'numParkFeaturesOpen', 'numSummerTrailsTotal', 'numSummerTrailsOpen', 'numSummerLiftsTotal', 'numSummerLiftsOpen'])
trails_total: 73
trails_open: 62
lifts_total: 14
lifts_open: 9
trail_report: Blue skies and fresh corduroy was the name of the game on the slopes today. Ready for more fun in the mountains on Tuesday? <br><br>We're planning to spin nine lifts across three peaks, giving you access to 62 trails for the day. First chair loads at 9:00am to get you up the hill and on to your favo
open_trails_pct: 85
open_lifts_pct: 64


In [6]:
loon_payload['comments']['comment'][0]['text']

'Blue skies and fresh corduroy was the name of the game on the slopes today. Ready for more fun in the mountains on Tuesday? <br><br>We\'re planning to spin nine lifts across three peaks, giving you access to 62 trails for the day. First chair loads at 9:00am to get you up the hill and on to your favorite terrain. The grooming team will spend the night making passes on most open trails, leaving behind that fresh corduroy you know and love for first tracks on Tuesday.<br><br>Gear up and get ready to ride. See you on the slopes.<br><br><b>Timbertown Quad</b><br>Timbertown Quad and terrain will be closed through the week and are scheduled to reopen Friday, March 6. Fox Run will remain open for access to the South Peak parking lot. Please note there will be no lift access.<br><br><b>Drop in.</b><br>Five parks are open with over 70 features, ready for you to drop in. Find large features and the 18\' Superpipe in Loon Mountain Park, medium features in Springboard and Little Sister, small fea

In [7]:
loon_payload['updated']

'2026-03-02T20:23:44Z'

Create into Function for Boyne Resorts

In [8]:
def safe_pct(open_val, total_val):
    if open_val is None or total_val in (None, 0):
        return None
    if open_val > total_val:
        return None
    return int(round((open_val / total_val) * 100))

create helper for report date

In [9]:
import pandas as pd

def to_report_date(ts_raw):
    if not ts_raw:
        return None
    ts = pd.to_datetime(ts_raw, utc=True)
    return ts.tz_convert("America/New_York").date()

Create Helper for epoch_to_iso

In [10]:
def epoch_to_iso(ts_raw):
    if ts_raw in (None, 0, ""):
        return None
    try:
        return pd.to_datetime(ts_raw, unit="s", utc=True).isoformat()
    except Exception:
        return None

Boyne Family parser

In [11]:
def parse_resort_family_boyne(payload: dict, resort: str = None, raw_path: str = None) -> dict:
    """
    Parser for Boyne-owned resort JSONs.

    Applicable resorts:
        - Loon
        - Sunday River
        - Sugarloaf

    Expected JSON structure:
        payload
            ├── updated (str)  <-- timestamp of resort conditions
            ├── currentConditions
            │     └── resortwide
            │           ├── numTrailsTotal
            │           ├── numTrailsOpen
            │           ├── numLiftsTotal
            │           └── numLiftsOpen
            └── comments
                  └── comment[0].text  <-- mountain report

    Extracted features:
        - report_date
        - resort_updated_at
        - trails_total
        - trails_open
        - open_trails_pct
        - lifts_total
        - lifts_open
        - open_lifts_pct
        - mountain_report_text

    Output schema:
        {
            "resort": str,
            "report_date": date | None,
            "raw_path": str | None,
            "resort_updated_at": str | None,
            "trails_open": int | None,
            "trails_total": int | None,
            "open_trails_pct": int | None,
            "lifts_open": int | None,
            "lifts_total": int | None,
            "open_lifts_pct": int | None,
            "mountain_report_text": str | None
        }

    Notes:
        - Uses safe_pct() to compute percentages safely
        - Uses payload['updated'] as the authoritative resort timestamp
        - report_date should be derived in local America/New_York date
        - Does not crash on missing fields (returns None instead)
    """

    resort_updated_at = payload.get("updated")
    report_date = to_report_date(resort_updated_at)

    current_conditions = payload.get("currentConditions", {}) or {}
    resortwide = current_conditions.get("resortwide", {}) or {}

    trails_total = resortwide.get("numTrailsTotal")
    trails_open = resortwide.get("numTrailsOpen")

    lifts_total = resortwide.get("numLiftsTotal")
    lifts_open = resortwide.get("numLiftsOpen")

    comments = payload.get("comments", {}) or {}
    comment_list = comments.get("comment", []) or []

    mountain_report_text = None
    if comment_list and isinstance(comment_list[0], dict):
        mountain_report_text = comment_list[0].get("text")

    open_trails_pct = safe_pct(trails_open, trails_total)
    open_lifts_pct = safe_pct(lifts_open, lifts_total)

    return {
        "resort": resort,
        "report_date": report_date,
        "raw_path": raw_path,
        "resort_updated_at": resort_updated_at,

        "trails_open": trails_open,
        "trails_total": trails_total,
        "open_trails_pct": open_trails_pct,

        "lifts_open": lifts_open,
        "lifts_total": lifts_total,
        "open_lifts_pct": open_lifts_pct,

        "mountain_report_text": mountain_report_text,
    }

Test Function for Loon

In [12]:
loon_fp=Path("data/raw/loon/Loon_reportpal_20260224_1855.json")
with open(loon_fp, "r") as f:
    loon_payload = json.load(f)
loon_row = parse_resort_family_boyne(loon_payload, "Loon", str(loon_fp))
print(loon_row)

{'resort': 'Loon', 'report_date': datetime.date(2026, 2, 24), 'raw_path': 'data/raw/loon/Loon_reportpal_20260224_1855.json', 'resort_updated_at': '2026-02-24T14:21:51Z', 'trails_open': 73, 'trails_total': 73, 'open_trails_pct': 100, 'lifts_open': 10, 'lifts_total': 14, 'open_lifts_pct': 71, 'mountain_report_text': 'Patrol is reporting 2 inches of fresh snow overnight, bringing our 24-hour total to 3 inches, and adding a little extra sparkle to an already picturesque summit. If you were waiting for a sign to get up here this is it. <br><br>Today brings slightly chillier temps than we\'ve seen lately, with highs around 16° at the summit and 24° at the base. Expect some gusty winds this morning, enough to keep things feeling extra crisp, before clouds gradually thin out as the day rolls on.<br><br>Overnight, the snowcats were out doing their thing, laying down fresh corduroy on most open terrain. Groomer lines are looking sharp, smooth, and ready for those first tracks.<br><br>Fresh snow.

### Test Function for Sunday River

In [13]:
sr_fp=Path("data/raw/sundayriver/SundayRiver_reportpal_20260224_1855.json")
with open(sr_fp, "r") as f:
    sr_payload = json.load(f)
sr_row = parse_resort_family_boyne(sr_payload, "SundayRiver", str(sr_fp))
print(sr_row)

{'resort': 'SundayRiver', 'report_date': datetime.date(2026, 2, 24), 'raw_path': 'data/raw/sundayriver/SundayRiver_reportpal_20260224_1855.json', 'resort_updated_at': '2026-02-24T18:38:40Z', 'trails_open': 143, 'trails_total': 144, 'open_trails_pct': 99, 'lifts_open': 14, 'lifts_total': 19, 'open_lifts_pct': 74, 'mountain_report_text': '<h5><br> Afternoon Report<br></h5><br><p><br> <strong>Last Updated: 2/24/26 | 1:40PM</strong><br><p><br> It would seem that the lesson for the last 24 hours is that weather can change on a dime. You know the saying for New England weather "if you don\'t like the weather, just wait 10 minutes and it\'ll change". Well today was another exercise in that. What started as an incredibly windy day turned into much more manageable gusts. Chairs came off wind hold pretty early this morning and we were able to operate our full fleet of lifts. Go figure.<br><p><br> Tomorrow looks far less windy and a little snowy. Nothing to get all revved up over, but again, any 

## Test For Sugarloaf

In [14]:
from pathlib import Path
import json

sl_fp = Path("data/raw/sugarloaf/Sugarloaf_reportpal_20260224_1855.json")

with open(sl_fp, "r") as f:
    sl_payload = json.load(f)

sl_row = parse_resort_family_boyne(sl_payload, "Sugarloaf", str(sl_fp))

print(sl_row)

{'resort': 'Sugarloaf', 'report_date': datetime.date(2026, 2, 24), 'raw_path': 'data/raw/sugarloaf/Sugarloaf_reportpal_20260224_1855.json', 'resort_updated_at': '2026-02-24T18:32:45Z', 'trails_open': 153, 'trails_total': 176, 'open_trails_pct': 87, 'lifts_open': 7, 'lifts_total': 15, 'open_lifts_pct': 47, 'mountain_report_text': "Morning *Update, 8:00 AM <br><br>Good morning, Sugarloafers.<br><br>That blizzard that body-slammed southern New England yesterday swung by on its way out and dropped off 2-3 inches of fresh snow.<br><br>The wind generously gathered all the snow into drifts that'll be super fun to find. Hint: check skier's left.<br><br>Winds are hanging around this morning but will gradually ease up through the day. The top of Skyline is reporting 4°F with northwest winds at 20 to 25 mph, gusting into the 30s. Skyline and SuperQuad look good to spin, but King Pine (*and Timberline) are on wind hold. Lift Ops is evaluating all other lifts. Keep an eye on lift status for live up

## Stratton and Sugarbush

## Testing for Stratton

In [15]:
from pathlib import Path
import json
from pprint import pprint

stratton_fp = Path("data/raw/stratton/Stratton_feed_20260224_1855.json")

with open(stratton_fp, "r") as f:
    stratton_payload = json.load(f)

print(stratton_payload.keys())

dict_keys(['LastUpdate', 'Resorts'])


In [16]:
stratton_payload['Resorts'][0]['SnowReport']['TotalLifts']
stratton_payload['Resorts'][0]['SnowReport']['TotalOpenLifts']
stratton_payload['Resorts'][0]['SnowReport']['TotalTrails']
stratton_payload['Resorts'][0]['SnowReport']['TotalOpenTrails']
stratton_payload['Resorts'][0]['SnowReport']['TotalOpenTrails']
stratton_payload['Resorts'][0]['SnowReport']['Report']



'<div><strong>2/24 5:45AM -</strong> Time to get up and at it for some Tuesday turns! It\'s getting real good out there after all the fresh snow this past week (16 inches to be exact) so come on up and get down on 96 open trails. Overnight, the grooming team laid fresh corduroy across 64 trails, leaving 32 all natural to hunt for pow stashes. Find your way up 8 lifts scheduled to spin at 9AM, with the possibility of wind related delays. Due to the Gondola being temporarily out of service, Shooting Star will run in its place today. Be sure to check out the most recent Gondola update below and stay tuned for more updates to come.</div><div><br></div><div>As temperatures dip, we are taking advantage of the cold window to fire up snow guns with focus on Upper Standard. Current trail closures are temporary and reflect ongoing snowmaking operations. Stay tuned for updates.</div><div><br></div><div>See you out there!&nbsp;</div><div><br></div><div><strong>Gondola Update 2/22:</strong></div><d

In [17]:
def parse_resort_family_resorts(payload: dict, resort: str = None, raw_path: str = None) -> dict:
    """
    Parser for resort JSONs with structure:
        payload['Resorts'][0]['SnowReport']

    Applicable resorts:
        - Stratton
        - Sugarbush

    Expected JSON structure:
        Resorts (list)
            └── [0]
                └── SnowReport (dict)
                    ├── TotalTrails
                    ├── TotalOpenTrails
                    ├── TotalLifts
                    ├── TotalOpenLifts
                    ├── Report
                    └── LastUpdate

    Extracted features:
        - report_date
        - resort_updated_at
        - trails_total
        - trails_open
        - open_trails_pct
        - lifts_total
        - lifts_open
        - open_lifts_pct
        - mountain_report_text

    Output schema:
        {
            "resort": str,
            "report_date": date | None,
            "raw_path": str | None,
            "resort_updated_at": str | None,
            "trails_open": int | None,
            "trails_total": int | None,
            "open_trails_pct": int | None,
            "lifts_open": int | None,
            "lifts_total": int | None,
            "open_lifts_pct": int | None,
            "mountain_report_text": str | None
        }

    Notes:
        - Uses safe_pct() to compute percentages safely
        - Assumes Resorts[0] contains the primary resort data
        - Uses SnowReport.LastUpdate as the authoritative timestamp
        - Does not crash on missing fields (returns None instead)
    """
    resorts = payload.get("Resorts", []) or []
    resort_obj = resorts[0] if resorts and isinstance(resorts[0], dict) else {}

    snow_report = resort_obj.get("SnowReport", {}) or {}

    trails_total = snow_report.get("TotalTrails")
    trails_open = snow_report.get("TotalOpenTrails")

    lifts_total = snow_report.get("TotalLifts")
    lifts_open = snow_report.get("TotalOpenLifts")

    mountain_report_text = snow_report.get("Report")

    resort_updated_at = snow_report.get("LastUpdate")
    report_date = to_report_date(resort_updated_at)

    return {
        "resort": resort,
        "report_date": report_date,
        "raw_path": raw_path,
        "resort_updated_at": resort_updated_at,

        "trails_open": trails_open,
        "trails_total": trails_total,
        "open_trails_pct": safe_pct(trails_open, trails_total),

        "lifts_open": lifts_open,
        "lifts_total": lifts_total,
        "open_lifts_pct": safe_pct(lifts_open, lifts_total),

        "mountain_report_text": mountain_report_text,
    }

In [18]:
stratton_row = parse_resort_family_resorts(stratton_payload, "Stratton", str(stratton_fp))
print(stratton_row)

{'resort': 'Stratton', 'report_date': datetime.date(2026, 2, 24), 'raw_path': 'data/raw/stratton/Stratton_feed_20260224_1855.json', 'resort_updated_at': '2026-02-24T05:44:33-0500', 'trails_open': 95, 'trails_total': 99, 'open_trails_pct': 96, 'lifts_open': 8, 'lifts_total': 14, 'open_lifts_pct': 57, 'mountain_report_text': '<div><strong>2/24 5:45AM -</strong> Time to get up and at it for some Tuesday turns! It\'s getting real good out there after all the fresh snow this past week (16 inches to be exact) so come on up and get down on 96 open trails. Overnight, the grooming team laid fresh corduroy across 64 trails, leaving 32 all natural to hunt for pow stashes. Find your way up 8 lifts scheduled to spin at 9AM, with the possibility of wind related delays. Due to the Gondola being temporarily out of service, Shooting Star will run in its place today. Be sure to check out the most recent Gondola update below and stay tuned for more updates to come.</div><div><br></div><div>As temperature

In [19]:
from pathlib import Path
import json

sugarbush_fp = Path("data/raw/sugarbush/Sugarbush_feed_20260224_1855.json")

with open(sugarbush_fp, "r") as f:
    sugarbush_payload = json.load(f)

In [20]:
sugarbush_row = parse_resort_family_resorts(
    sugarbush_payload,
    "Sugarbush",
    str(sugarbush_fp)
)

print(sugarbush_row)

{'resort': 'Sugarbush', 'report_date': datetime.date(2026, 2, 24), 'raw_path': 'data/raw/sugarbush/Sugarbush_feed_20260224_1855.json', 'resort_updated_at': '2026-02-24T13:21:19-0500', 'trails_open': 111, 'trails_total': 111, 'open_trails_pct': 100, 'lifts_open': 14, 'lifts_total': 16, 'open_lifts_pct': 88, 'mountain_report_text': '<div>February 24th, 1:21 PM<br>*Super Bravo is on maintenance delay and Valley House will run in its place*<br>*GMX will be closed for the day for maintenance and Inverness will run in its place*<br>Good afternoon riders and skiers! Come tap into the mid-week magic at the Bush!&nbsp; The forecast calls for partly cloudy skies and temps in the upper teens at the base and upper single digits at the summit with a clipper system heading our way, poised to deliver a couple inches through tomorrow.&nbsp; There are still stashes to be found amid the 453 skiable acres, 13 lifts, 111 open trails, and 50 refreshed groomers including fan favorites Ripcord, Elbow, and Je

## Killington Pico Parser

For killington , we have 3 files , lifts (which contain lift status data), snow_reports (Daily mountain report and snow totals), and and trails (trail status). <br> 
We need to consolidate this data in order to have a single row in our resort data to feed into our rag.

## Killington lifts

In [21]:
def parse_killington_lifts(lifts_payload: list) -> dict:
    """
    Parse Killington/Pico lifts JSON.

    Rules:
        - lifts_total = count of all lift objects
        - lifts_open = count where status in {'open', 'expected'}
        - lifts_updated_at = latest updated timestamp across lift objects
    """
    lifts_payload = lifts_payload or []

    lifts_total = len(lifts_payload)
    lifts_open = sum(
        1 for x in lifts_payload
        if (x.get("status") or "").lower() in {"open", "expected"}
    )

    updated_vals = [
        x.get("updated") for x in lifts_payload
        if x.get("updated") not in (None, 0, "")
    ]
    lifts_updated_at = epoch_to_iso(max(updated_vals)) if updated_vals else None

    return {
        "lifts_open": lifts_open,
        "lifts_total": lifts_total,
        "open_lifts_pct": safe_pct(lifts_open, lifts_total),
        "lifts_updated_at": lifts_updated_at,
    }

def parse_killington_trails(trails_payload: list) -> dict:
    """
    Parse Killington/Pico trails JSON.

    Rules:
        - only count trails where:
            season == 'winter'
            type == 'alpine_trail'
            include == True
        - trails_total = count of filtered trail objects
        - trails_open = count of filtered trail objects where status == 'open'
        - trails_updated_at = latest updated timestamp across filtered trail objects
    """
    trails_payload = trails_payload or []

    filtered_trails = [
        x for x in trails_payload
        if (x.get("season") == "winter")
        and (x.get("type") == "alpine_trail")
        and (x.get("include") is True)
    ]

    trails_total = len(filtered_trails)
    trails_open = sum(
        1 for x in filtered_trails
        if (x.get("status") or "").lower() == "open"
    )

    updated_vals = [
        x.get("updated") for x in filtered_trails
        if x.get("updated") not in (None, 0, "")
    ]
    trails_updated_at = epoch_to_iso(max(updated_vals)) if updated_vals else None

    return {
        "trails_open": trails_open,
        "trails_total": trails_total,
        "open_trails_pct": safe_pct(trails_open, trails_total),
        "trails_updated_at": trails_updated_at,
    }

def parse_killington_snow_report(snow_payload: list) -> dict:
    """
    Parse Killington/Pico snow report JSON.

    Assumes:
        - payload is a list of report objects
        - the first object payload[0] is the report to use
    """
    snow_payload = snow_payload or []
    report_obj = snow_payload[0] if snow_payload and isinstance(snow_payload[0], dict) else {}

    mountain_report_text = (
        report_obj.get("report")
        or report_obj.get("Report")
        or report_obj.get("text")
    )

    updated_raw = report_obj.get("updated") or report_obj.get("Updated")
    snow_report_updated_at = epoch_to_iso(updated_raw)

    return {
        "mountain_report_text": mountain_report_text,
        "snow_report_updated_at": snow_report_updated_at,
    }

def parse_resort_family_killington(
    lifts_payload: list,
    trails_payload: list,
    snow_payload: list,
    resort: str = None,
    raw_path: str = None
) -> dict:
    """
    Parser for Killington/Pico resort data split across 3 JSON files:
        - lifts
        - trails
        - snow reports

    Applicable resorts:
        - Killington
        - Pico

    Output schema matches consolidated resort_status_daily table:
        {
            "resort": str,
            "report_date": date | None,
            "raw_path": str | None,
            "resort_updated_at": str | None,
            "trails_open": int | None,
            "trails_total": int | None,
            "open_trails_pct": int | None,
            "lifts_open": int | None,
            "lifts_total": int | None,
            "open_lifts_pct": int | None,
            "mountain_report_text": str | None
        }

    Notes:
        - Uses latest timestamp across lifts/trails/snow report as resort_updated_at
        - Assumes lift status in {'open', 'expected'} counts as open
        - Assumes only winter alpine trails with include=True count toward trail totals
    """
    lift_info = parse_killington_lifts(lifts_payload)
    trail_info = parse_killington_trails(trails_payload)
    snow_info = parse_killington_snow_report(snow_payload)

    candidate_ts = [
        lift_info.get("lifts_updated_at"),
        trail_info.get("trails_updated_at"),
        snow_info.get("snow_report_updated_at"),
    ]
    candidate_ts = [x for x in candidate_ts if x]

    resort_updated_at = max(candidate_ts) if candidate_ts else None
    report_date = to_report_date(resort_updated_at)

    return {
        "resort": resort,
        "report_date": report_date,
        "raw_path": raw_path,
        "resort_updated_at": resort_updated_at,

        "trails_open": trail_info.get("trails_open"),
        "trails_total": trail_info.get("trails_total"),
        "open_trails_pct": trail_info.get("open_trails_pct"),

        "lifts_open": lift_info.get("lifts_open"),
        "lifts_total": lift_info.get("lifts_total"),
        "open_lifts_pct": lift_info.get("open_lifts_pct"),

        "mountain_report_text": snow_info.get("mountain_report_text"),
    }

In [22]:
from pathlib import Path
import json

kill_lifts_fp = Path("data/raw/killington/killington_lifts_20260224_1855.json")
kill_trails_fp = Path("data/raw/killington/killington_trails_20260224_1855.json")
kill_snow_fp = Path("data/raw/killington/killington_snow_reports_20260224_1855.json")

with open(kill_lifts_fp, "r") as f:
    kill_lifts_payload = json.load(f)

with open(kill_trails_fp, "r") as f:
    kill_trails_payload = json.load(f)

with open(kill_snow_fp, "r") as f:
    kill_snow_payload = json.load(f)

kill_row = parse_resort_family_killington(
    lifts_payload=kill_lifts_payload,
    trails_payload=kill_trails_payload,
    snow_payload=kill_snow_payload,
    resort="Killington",
    raw_path=" | ".join([str(kill_lifts_fp), str(kill_trails_fp), str(kill_snow_fp)])
)

print(kill_row)

{'resort': 'Killington', 'report_date': datetime.date(2026, 2, 24), 'raw_path': 'data/raw/killington/killington_lifts_20260224_1855.json | data/raw/killington/killington_trails_20260224_1855.json | data/raw/killington/killington_snow_reports_20260224_1855.json', 'resort_updated_at': '2026-02-24T16:51:15+00:00', 'trails_open': 154, 'trails_total': 155, 'open_trails_pct': 99, 'lifts_open': 13, 'lifts_total': 19, 'open_lifts_pct': 68, 'mountain_report_text': '<p>Good morning, Beast fans!</p><p>After receiving some of the outskirts of yesterday\'s Nor\'easter, today is lining up to be another prime midwinter day out there. Another 2" of snowfall gave us a slight refresh on surfaces and kept turns feeling soft yesterday.</p><p>Today trends colder than yesterday, with highs around 8 degrees at the summit and in the teens at the base. Visibility is pretty low to start the day, so make sure to control your speed while you\'re out there. A north breeze will start the day on the stronger side be

Found 1 error in killington trails , missmatch between the mountain report and actual data. 

In [23]:
def debug_killington_trails(trails_payload: list):
    """
    Debug helper to inspect Killington trails filtering and open counts.
    """

    trails_payload = trails_payload or []

    print(f"Total raw trails: {len(trails_payload)}\n")

    # -----------------------------
    # Apply same filter as parser
    # -----------------------------
    filtered = [
        t for t in trails_payload
        if (t.get("season") == "winter")
        and (t.get("type") == "alpine_trail")
        and (t.get("include") is True)
    ]

    print(f"Filtered trails (valid for skiing): {len(filtered)}\n")

    # -----------------------------
    # Separate open vs closed
    # -----------------------------
    open_trails = [
        t for t in filtered
        if (t.get("status") or "").lower() == "open"
    ]

    closed_trails = [
        t for t in filtered
        if (t.get("status") or "").lower() != "open"
    ]

    print(f"Open trails: {len(open_trails)}")
    print(f"Closed trails: {len(closed_trails)}\n")

    # -----------------------------
    # Print problematic trails
    # -----------------------------
    print("Closed / non-open trails:\n")

    for t in closed_trails:
        print({
            "name": t.get("name"),
            "status": t.get("status"),
            "season": t.get("season"),
            "type": t.get("type"),
            "include": t.get("include"),
            "difficulty": t.get("difficulty")
        })

    # -----------------------------
    # Optional: sanity check %
    # -----------------------------
    total = len(filtered)
    open_count = len(open_trails)

    if total:
        pct = round((open_count / total) * 100)
        print(f"\nComputed % open: {pct}%")

In [24]:
debug_killington_trails(kill_trails_payload)

Total raw trails: 222

Filtered trails (valid for skiing): 155

Open trails: 154
Closed trails: 1

Closed / non-open trails:

{'name': 'Blue Heaven', 'status': 'closed', 'season': 'winter', 'type': 'alpine_trail', 'include': True, 'difficulty': 'more_difficult'}

Computed % open: 99%


Blue heaven is marked as closed on the API , but is open according to the mountain report. this is most likely an error on killington's end

In [25]:
from pathlib import Path
import json
import pandas as pd

# -----------------------------
# Pico file paths
# -----------------------------
pico_lifts_fp = Path("data/raw/pico/Pico_lifts_20260224_1855.json")
pico_trails_fp = Path("data/raw/pico/Pico_trails_20260224_1855.json")
pico_snow_fp = Path("data/raw/pico/Pico_snow_reports_20260224_1855.json")

# -----------------------------
# Load JSON payloads
# -----------------------------
with open(pico_lifts_fp, "r") as f:
    pico_lifts_payload = json.load(f)

with open(pico_trails_fp, "r") as f:
    pico_trails_payload = json.load(f)

with open(pico_snow_fp, "r") as f:
    pico_snow_payload = json.load(f)

# -----------------------------
# Parse consolidated Pico row
# -----------------------------
pico_row = parse_resort_family_killington(
    lifts_payload=pico_lifts_payload,
    trails_payload=pico_trails_payload,
    snow_payload=pico_snow_payload,
    resort="Pico",
    raw_path=" | ".join([
        str(pico_lifts_fp),
        str(pico_trails_fp),
        str(pico_snow_fp)
    ])
)

print(pico_row)

{'resort': 'Pico', 'report_date': datetime.date(2026, 2, 24), 'raw_path': 'data/raw/pico/Pico_lifts_20260224_1855.json | data/raw/pico/Pico_trails_20260224_1855.json | data/raw/pico/Pico_snow_reports_20260224_1855.json', 'resort_updated_at': '2026-02-24T16:35:52+00:00', 'trails_open': 56, 'trails_total': 57, 'open_trails_pct': 98, 'lifts_open': 4, 'lifts_total': 6, 'open_lifts_pct': 67, 'mountain_report_text': '<p>Good morning, Pico family,</p><p>After some milder weather in recent times, today will feel a little more like February again. Yesterday brought another 2" of snow, another surface refresh that kept things feeling fresh out there.</p><p>High temps today will reach around 10 degrees at the summit and stay in the teens at the base. A north/northwest breeze will be noticeable early before settling down during the day. It’ll feel a bit chilly early this morning, so break out your midwinter gear and face covers and you\'ll be all set.</p><p>Surfaces remain in a great spot. Packed 

In [26]:
import pandas as pd

EXPECTED_COLUMNS = [
    "resort",
    "report_date",
    "raw_path",
    "resort_updated_at",
    "trails_open",
    "trails_total",
    "open_trails_pct",
    "lifts_open",
    "lifts_total",
    "open_lifts_pct",
    "mountain_report_text",
]

all_rows = [
    loon_row,
    sr_row,
    sl_row,
    stratton_row,
    sugarbush_row,
    kill_row,
    pico_row,
]

df_all_resorts = pd.DataFrame(all_rows)

print(df_all_resorts.columns.tolist())
print(df_all_resorts[EXPECTED_COLUMNS])

['resort', 'report_date', 'raw_path', 'resort_updated_at', 'trails_open', 'trails_total', 'open_trails_pct', 'lifts_open', 'lifts_total', 'open_lifts_pct', 'mountain_report_text']
        resort report_date                                           raw_path  \
0         Loon  2026-02-24    data/raw/loon/Loon_reportpal_20260224_1855.json   
1  SundayRiver  2026-02-24  data/raw/sundayriver/SundayRiver_reportpal_202...   
2    Sugarloaf  2026-02-24  data/raw/sugarloaf/Sugarloaf_reportpal_2026022...   
3     Stratton  2026-02-24  data/raw/stratton/Stratton_feed_20260224_1855....   
4    Sugarbush  2026-02-24  data/raw/sugarbush/Sugarbush_feed_20260224_185...   
5   Killington  2026-02-24  data/raw/killington/killington_lifts_20260224_...   
6         Pico  2026-02-24  data/raw/pico/Pico_lifts_20260224_1855.json | ...   

           resort_updated_at  trails_open  trails_total  open_trails_pct  \
0       2026-02-24T14:21:51Z           73            73              100   
1       2026-02-24T

Load loon data into DB

In [31]:
from pathlib import Path
import json
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

def load_loon_reportpal_rows(folder: str = "data/raw/loon") -> pd.DataFrame:
    folder_path = Path(folder)

    files = sorted(folder_path.glob("Loon_reportpal_*.json"))
    print(f"Found {len(files)} Loon reportpal files")

    rows = []
    for fp in files:
        with open(fp, "r") as f:
            payload = json.load(f)

        row = parse_resort_family_boyne(
            payload=payload,
            resort="Loon",
            raw_path=str(fp)
        )
        rows.append(row)

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df[
            [
                "resort",
                "report_date",
                "raw_path",
                "resort_updated_at",
                "trails_open",
                "trails_total",
                "open_trails_pct",
                "lifts_open",
                "lifts_total",
                "open_lifts_pct",
                "mountain_report_text",
            ]
        ]

    return df

def upsert_resort_status_rows(df: pd.DataFrame):
    if df.empty:
        print("No rows to insert.")
        return

    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        dbname="ski_pipeline",
        user="airflow",
        password="airflow",
    )
    conn.autocommit = False

    sql = """
    INSERT INTO resort_status_daily (
        resort,
        report_date,
        raw_path,
        resort_updated_at,
        trails_open,
        trails_total,
        open_trails_pct,
        lifts_open,
        lifts_total,
        open_lifts_pct,
        mountain_report_text
    )
    VALUES %s
    ON CONFLICT (resort, report_date, resort_updated_at)
    DO NOTHING;
    """

    values = [
        (
            row["resort"],
            row["report_date"],
            row["raw_path"],
            row["resort_updated_at"],
            row["trails_open"],
            row["trails_total"],
            row["open_trails_pct"],
            row["lifts_open"],
            row["lifts_total"],
            row["open_lifts_pct"],
            row["mountain_report_text"],
        )
        for _, row in df.iterrows()
    ]

    try:
        with conn.cursor() as cur:
            execute_values(cur, sql, values, page_size=100)
        conn.commit()
        print(f"Inserted/upserted {len(values)} rows.")
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

In [32]:
df_loon = load_loon_reportpal_rows("data/raw/loon")
print(df_loon.head())
print(df_loon.shape)


Found 71 Loon reportpal files
  resort report_date                                         raw_path  \
0   Loon  2026-02-16  data/raw/loon/Loon_reportpal_20260217_0302.json   
1   Loon  2026-02-16  data/raw/loon/Loon_reportpal_20260217_0314.json   
2   Loon  2026-02-17  data/raw/loon/Loon_reportpal_20260217_1208.json   
3   Loon  2026-02-17  data/raw/loon/Loon_reportpal_20260218_0012.json   
4   Loon  2026-02-18  data/raw/loon/Loon_reportpal_20260218_1207.json   

      resort_updated_at  trails_open  trails_total  open_trails_pct  \
0  2026-02-16T19:49:09Z           73            73              100   
1  2026-02-16T19:49:09Z           73            73              100   
2  2026-02-17T12:07:01Z           73            73              100   
3  2026-02-17T20:28:15Z           73            73              100   
4  2026-02-18T11:52:52Z           73            73              100   

   lifts_open  lifts_total  open_lifts_pct  \
0          12           14              86   
1          1

In [33]:
upsert_resort_status_rows(df_loon)

Inserted/upserted 71 rows.


In [35]:
from pathlib import Path
import json
import pandas as pd

def load_boyne_reportpal_rows(resort: str, folder: str, pattern: str) -> pd.DataFrame:
    folder_path = Path(folder)
    files = sorted(folder_path.glob(pattern))
    print(f"Found {len(files)} files for {resort}")

    rows = []
    for fp in files:
        with open(fp, "r") as f:
            payload = json.load(f)

        row = parse_resort_family_boyne(
            payload=payload,
            resort=resort,
            raw_path=str(fp)
        )
        rows.append(row)

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df[
            [
                "resort",
                "report_date",
                "raw_path",
                "resort_updated_at",
                "trails_open",
                "trails_total",
                "open_trails_pct",
                "lifts_open",
                "lifts_total",
                "open_lifts_pct",
                "mountain_report_text",
            ]
        ]

        df = df.drop_duplicates(
            subset=["resort", "report_date", "resort_updated_at"]
        )

    return df

In [36]:
df_sr = load_boyne_reportpal_rows(
    resort="SundayRiver",
    folder="data/raw/sundayriver",
    pattern="SundayRiver_reportpal_*.json"
)

print(df_sr.head())
print(df_sr.shape)

Found 69 files for SundayRiver
        resort report_date                                           raw_path  \
0  SundayRiver  2026-02-16  data/raw/sundayriver/SundayRiver_reportpal_202...   
2  SundayRiver  2026-02-17  data/raw/sundayriver/SundayRiver_reportpal_202...   
3  SundayRiver  2026-02-17  data/raw/sundayriver/SundayRiver_reportpal_202...   
4  SundayRiver  2026-02-18  data/raw/sundayriver/SundayRiver_reportpal_202...   
5  SundayRiver  2026-02-18  data/raw/sundayriver/SundayRiver_reportpal_202...   

      resort_updated_at  trails_open  trails_total  open_trails_pct  \
0  2026-02-16T21:03:17Z          143           144               99   
2  2026-02-17T11:51:04Z          142           144               99   
3  2026-02-17T20:58:36Z          142           144               99   
4  2026-02-18T11:28:26Z          142           144               99   
5  2026-02-18T20:56:51Z          142           144               99   

   lifts_open  lifts_total  open_lifts_pct  \
0        

In [37]:
df_sr = load_boyne_reportpal_rows(
    resort="SundayRiver",
    folder="data/raw/sundayriver",
    pattern="SundayRiver_reportpal_*.json"
)

print(df_sr.head())
print(df_sr.shape)

Found 69 files for SundayRiver
        resort report_date                                           raw_path  \
0  SundayRiver  2026-02-16  data/raw/sundayriver/SundayRiver_reportpal_202...   
2  SundayRiver  2026-02-17  data/raw/sundayriver/SundayRiver_reportpal_202...   
3  SundayRiver  2026-02-17  data/raw/sundayriver/SundayRiver_reportpal_202...   
4  SundayRiver  2026-02-18  data/raw/sundayriver/SundayRiver_reportpal_202...   
5  SundayRiver  2026-02-18  data/raw/sundayriver/SundayRiver_reportpal_202...   

      resort_updated_at  trails_open  trails_total  open_trails_pct  \
0  2026-02-16T21:03:17Z          143           144               99   
2  2026-02-17T11:51:04Z          142           144               99   
3  2026-02-17T20:58:36Z          142           144               99   
4  2026-02-18T11:28:26Z          142           144               99   
5  2026-02-18T20:56:51Z          142           144               99   

   lifts_open  lifts_total  open_lifts_pct  \
0        

In [38]:
upsert_resort_status_rows(df_sr)

Inserted/upserted 49 rows.


In [39]:
df_sl = load_boyne_reportpal_rows(
    resort="Sugarloaf",
    folder="data/raw/sugarloaf",
    pattern="Sugarloaf_reportpal_*.json"
)
upsert_resort_status_rows(df_sl)

Found 69 files for Sugarloaf
Inserted/upserted 64 rows.


Loader for Resorts Family

In [40]:
from pathlib import Path
import json
import pandas as pd

def load_resorts_feed_rows(resort: str, folder: str, pattern: str) -> pd.DataFrame:
    folder_path = Path(folder)
    files = sorted(folder_path.glob(pattern))
    print(f"Found {len(files)} files for {resort}")

    rows = []
    for fp in files:
        with open(fp, "r") as f:
            payload = json.load(f)

        row = parse_resort_family_resorts(
            payload=payload,
            resort=resort,
            raw_path=str(fp)
        )
        rows.append(row)

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df[
            [
                "resort",
                "report_date",
                "raw_path",
                "resort_updated_at",
                "trails_open",
                "trails_total",
                "open_trails_pct",
                "lifts_open",
                "lifts_total",
                "open_lifts_pct",
                "mountain_report_text",
            ]
        ]

        df = df.drop_duplicates(
            subset=["resort", "report_date", "resort_updated_at"]
        )

    return df

In [41]:
df_stratton = load_resorts_feed_rows(
    resort="Stratton",
    folder="data/raw/stratton",
    pattern="Stratton_feed_*.json"
)

print(df_stratton.head())
print(df_stratton.shape)

upsert_resort_status_rows(df_stratton)

Found 69 files for Stratton
     resort report_date                                           raw_path  \
0  Stratton  2026-02-16  data/raw/stratton/Stratton_feed_20260217_0302....   
2  Stratton  2026-02-17  data/raw/stratton/Stratton_feed_20260217_1208....   
3  Stratton  2026-02-17  data/raw/stratton/Stratton_feed_20260218_0012....   
4  Stratton  2026-02-18  data/raw/stratton/Stratton_feed_20260218_1207....   
5  Stratton  2026-02-18  data/raw/stratton/Stratton_feed_20260219_0000....   

          resort_updated_at  trails_open  trails_total  open_trails_pct  \
0  2026-02-16T21:34:40-0500           95            99               96   
2  2026-02-17T06:02:48-0500           95            99               96   
3  2026-02-17T15:04:58-0500           94            99               95   
4  2026-02-18T05:44:19-0500           94            99               95   
5  2026-02-18T14:28:17-0500           97            99               98   

   lifts_open  lifts_total  open_lifts_pct  \
0     

In [42]:
df_sugarbush = load_resorts_feed_rows(
    resort="Sugarbush",
    folder="data/raw/sugarbush",
    pattern="Sugarbush_feed_*.json"
)

print(df_sugarbush.head())
print(df_sugarbush.shape)

upsert_resort_status_rows(df_sugarbush)

Found 69 files for Sugarbush
      resort report_date                                           raw_path  \
0  Sugarbush  2026-02-16  data/raw/sugarbush/Sugarbush_feed_20260217_030...   
2  Sugarbush  2026-02-17  data/raw/sugarbush/Sugarbush_feed_20260217_120...   
3  Sugarbush  2026-02-17  data/raw/sugarbush/Sugarbush_feed_20260218_001...   
4  Sugarbush  2026-02-18  data/raw/sugarbush/Sugarbush_feed_20260218_120...   
5  Sugarbush  2026-02-18  data/raw/sugarbush/Sugarbush_feed_20260219_000...   

          resort_updated_at  trails_open  trails_total  open_trails_pct  \
0  2026-02-16T13:22:47-0500          111           111              100   
2  2026-02-17T07:08:33-0500          111           111              100   
3  2026-02-17T13:47:49-0500          110           111               99   
4  2026-02-18T07:01:06-0500          109           111               98   
5  2026-02-18T15:34:45-0500          109           111               98   

   lifts_open  lifts_total  open_lifts_pct  \

Killinton Pico

In [43]:
from pathlib import Path
import json
import pandas as pd
import re

def extract_file_timestamp(fp: Path):
    """
    Extract timestamp token like 20260224_1855 from file name.
    Returns None if no match is found.
    """
    m = re.search(r"(\d{8}_\d{4})\.json$", fp.name)
    return m.group(1) if m else None

In [44]:
def load_killington_family_rows(resort: str, folder: str, prefix: str) -> pd.DataFrame:
    """
    Load all Killington/Pico family rows from a folder.

    Expected files per timestamp:
        {prefix}_lifts_YYYYMMDD_HHMM.json
        {prefix}_trails_YYYYMMDD_HHMM.json
        {prefix}_snow_reports_YYYYMMDD_HHMM.json

    Example prefix:
        - killington
        - Pico   (adjust based on actual file naming)
    """
    folder_path = Path(folder)

    lift_files = sorted(folder_path.glob(f"{prefix}_lifts_*.json"))
    trail_files = sorted(folder_path.glob(f"{prefix}_trails_*.json"))
    snow_files = sorted(folder_path.glob(f"{prefix}_snow_reports_*.json"))

    print(f"{resort}: found {len(lift_files)} lifts, {len(trail_files)} trails, {len(snow_files)} snow report files")

    lift_map = {extract_file_timestamp(fp): fp for fp in lift_files if extract_file_timestamp(fp)}
    trail_map = {extract_file_timestamp(fp): fp for fp in trail_files if extract_file_timestamp(fp)}
    snow_map = {extract_file_timestamp(fp): fp for fp in snow_files if extract_file_timestamp(fp)}

    common_keys = sorted(set(lift_map) & set(trail_map) & set(snow_map))
    print(f"{resort}: found {len(common_keys)} complete timestamp groups")

    rows = []

    for ts_key in common_keys:
        lifts_fp = lift_map[ts_key]
        trails_fp = trail_map[ts_key]
        snow_fp = snow_map[ts_key]

        with open(lifts_fp, "r") as f:
            lifts_payload = json.load(f)

        with open(trails_fp, "r") as f:
            trails_payload = json.load(f)

        with open(snow_fp, "r") as f:
            snow_payload = json.load(f)

        row = parse_resort_family_killington(
            lifts_payload=lifts_payload,
            trails_payload=trails_payload,
            snow_payload=snow_payload,
            resort=resort,
            raw_path=" | ".join([str(lifts_fp), str(trails_fp), str(snow_fp)])
        )
        rows.append(row)

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df[
            [
                "resort",
                "report_date",
                "raw_path",
                "resort_updated_at",
                "trails_open",
                "trails_total",
                "open_trails_pct",
                "lifts_open",
                "lifts_total",
                "open_lifts_pct",
                "mountain_report_text",
            ]
        ]

        df = df.drop_duplicates(
            subset=["resort", "report_date", "resort_updated_at"]
        )

    return df

In [46]:
df_killington = load_killington_family_rows(
    resort="Killington",
    folder="data/raw/killington",
    prefix="Killington"
)

print(df_killington.head())
print(df_killington.shape)

upsert_resort_status_rows(df_killington)

Killington: found 69 lifts, 69 trails, 69 snow report files
Killington: found 67 complete timestamp groups
       resort report_date                                           raw_path  \
0  Killington  2026-02-16  data/raw/killington/Killington_lifts_20260217_...   
2  Killington  2026-02-17  data/raw/killington/Killington_lifts_20260217_...   
3  Killington  2026-02-17  data/raw/killington/Killington_lifts_20260218_...   
4  Killington  2026-02-18  data/raw/killington/Killington_lifts_20260218_...   
5  Killington  2026-02-18  data/raw/killington/Killington_lifts_20260219_...   

           resort_updated_at  trails_open  trails_total  open_trails_pct  \
0  2026-02-17T01:53:53+00:00          155           155              100   
2  2026-02-17T11:21:36+00:00          155           155              100   
3  2026-02-17T23:13:28+00:00          155           155              100   
4  2026-02-18T11:05:08+00:00          155           155              100   
5  2026-02-18T21:19:45+00:00    

In [47]:
df_pico = load_killington_family_rows(
    resort="Pico",
    folder="data/raw/pico",
    prefix="Pico"
)

print(df_pico.head())
print(df_pico.shape)

upsert_resort_status_rows(df_pico)

Pico: found 69 lifts, 69 trails, 69 snow report files
Pico: found 67 complete timestamp groups
  resort report_date                                           raw_path  \
0   Pico  2026-02-16  data/raw/pico/Pico_lifts_20260217_0302.json | ...   
2   Pico  2026-02-17  data/raw/pico/Pico_lifts_20260217_1208.json | ...   
3   Pico  2026-02-17  data/raw/pico/Pico_lifts_20260218_0012.json | ...   
4   Pico  2026-02-18  data/raw/pico/Pico_lifts_20260218_1207.json | ...   
5   Pico  2026-02-18  data/raw/pico/Pico_lifts_20260219_0000.json | ...   

           resort_updated_at  trails_open  trails_total  open_trails_pct  \
0  2026-02-16T21:02:29+00:00           57            57              100   
2  2026-02-17T11:43:57+00:00           57            57              100   
3  2026-02-17T23:35:56+00:00           57            57              100   
4  2026-02-18T11:57:44+00:00           57            57              100   
5  2026-02-18T21:36:03+00:00           57            57              100  

## Open Meteo Data

In [54]:
def extract_forecast_run_at_from_filename(fp: str | Path):
    """
    Extract forecast run timestamp from filename like:
    Pico_forecast_20260224_1610.json
    """
    fp = Path(fp)
    m = re.search(r"(\d{8}_\d{4})\.json$", fp.name)
    if not m:
        return None

    ts = pd.to_datetime(m.group(1), format="%Y%m%d_%H%M")
    return ts.tz_localize("America/New_York").tz_convert("UTC").isoformat()
def rain_proxy_mm(temp_c, precip_mm):
    if temp_c is None or precip_mm is None:
        return 0.0
    try:
        return float(precip_mm) if float(temp_c) > 0 and float(precip_mm) > 0 else 0.0
    except Exception:
        return 0.0
def count_freeze_thaw_cycles(series_temp_c: pd.Series) -> int:
    """
    Count crossings around 0C between consecutive hours.
    """
    s = series_temp_c.dropna().reset_index(drop=True)
    if len(s) < 2:
        return 0

    prev = s.shift(1)
    crossings = ((prev <= 0) & (s > 0)) | ((prev > 0) & (s <= 0))
    return int(crossings.sum())

In [55]:
def parse_openmeteo_hourly(payload: dict, resort: str, raw_path: str) -> pd.DataFrame:
    """
    Parse one Open-Meteo forecast JSON into normalized hourly rows.

    Output grain:
        resort + forecast_run_at + time_local
    """
    forecast_run_at = extract_forecast_run_at_from_filename(raw_path)
    timezone_name = payload.get("timezone", "America/New_York")

    hourly = payload.get("hourly", {}) or {}

    df = pd.DataFrame({
        "time_raw": hourly.get("time", []) or [],
        "temperature_2m_c": hourly.get("temperature_2m", []) or [],
        "cloudcover_pct": hourly.get("cloudcover", []) or [],
        "wind_speed_10m_kmh": hourly.get("wind_speed_10m", []) or [],
        "wind_gusts_10m_kmh": hourly.get("wind_gusts_10m", []) or [],
        "precipitation_mm": hourly.get("precipitation", []) or [],
        "snowfall_cm": hourly.get("snowfall", []) or [],
    })

    if df.empty:
        return df

    # Open-Meteo sample uses local timezone strings already
    df["time_local"] = pd.to_datetime(df["time_raw"])
    if df["time_local"].dt.tz is None:
        df["time_local"] = df["time_local"].dt.tz_localize(timezone_name)

    df["resort"] = resort
    df["forecast_run_at"] = forecast_run_at
    df["target_ski_date"] = df["time_local"].dt.date
    df["hour_local"] = df["time_local"].dt.hour

    df = df[
        [
            "resort",
            "forecast_run_at",
            "time_local",
            "target_ski_date",
            "hour_local",
            "temperature_2m_c",
            "cloudcover_pct",
            "wind_speed_10m_kmh",
            "wind_gusts_10m_kmh",
            "precipitation_mm",
            "snowfall_cm",
        ]
    ].copy()

    return df

In [56]:
def build_openmeteo_daily_features(hourly_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build daily ski features from normalized hourly Open-Meteo data.

    Grain:
        resort + forecast_run_at + target_ski_date
    """
    if hourly_df.empty:
        return pd.DataFrame(columns=[
            "resort", "forecast_run_at", "target_ski_date",
            "avg_temp_ski_hours_c", "min_temp_ski_hours_c", "max_temp_ski_hours_c",
            "avg_cloudcover_ski_hours_pct", "avg_wind_speed_ski_hours_kmh", "max_wind_gust_ski_hours_kmh",
            "snowfall_ski_day_cm", "rain_ski_day_mm",
            "snowfall_prev_day_cm", "rain_prev_day_mm", "hours_above_freezing_prev_day",
            "freeze_thaw_cycles_prev_48h", "ice_risk_flag", "wind_hold_risk_flag"
        ])

    df = hourly_df.copy()

    # rain proxy
    df["rain_proxy_mm"] = df.apply(
        lambda row: rain_proxy_mm(row["temperature_2m_c"], row["precipitation_mm"]),
        axis=1
    )

    # ski hours: 8-16 inclusive
    ski_df = df[(df["hour_local"] >= 8) & (df["hour_local"] <= 16)].copy()

    ski_day = (
        ski_df.groupby(["resort", "forecast_run_at", "target_ski_date"], as_index=False)
        .agg(
            avg_temp_ski_hours_c=("temperature_2m_c", "mean"),
            min_temp_ski_hours_c=("temperature_2m_c", "min"),
            max_temp_ski_hours_c=("temperature_2m_c", "max"),
            avg_cloudcover_ski_hours_pct=("cloudcover_pct", "mean"),
            avg_wind_speed_ski_hours_kmh=("wind_speed_10m_kmh", "mean"),
            max_wind_gust_ski_hours_kmh=("wind_gusts_10m_kmh", "max"),
            snowfall_ski_day_cm=("snowfall_cm", "sum"),
            rain_ski_day_mm=("rain_proxy_mm", "sum"),
        )
    )

    # full-day aggregates for prior-day features
    full_day = (
        df.groupby(["resort", "forecast_run_at", "target_ski_date"], as_index=False)
        .agg(
            snowfall_day_total_cm=("snowfall_cm", "sum"),
            rain_day_total_mm=("rain_proxy_mm", "sum"),
            hours_above_freezing_day=("temperature_2m_c", lambda s: int((s > 0).sum())),
        )
    )

    results = []

    for _, row in ski_day.iterrows():
        resort = row["resort"]
        forecast_run_at = row["forecast_run_at"]
        target_ski_date = row["target_ski_date"]

        prev_day = target_ski_date - pd.Timedelta(days=1)
        prev_48h_start = pd.Timestamp(target_ski_date).tz_localize("America/New_York") - pd.Timedelta(hours=48)
        prev_48h_end = pd.Timestamp(target_ski_date).tz_localize("America/New_York")

        # prior day stats
        prior_match = full_day[
            (full_day["resort"] == resort) &
            (full_day["forecast_run_at"] == forecast_run_at) &
            (full_day["target_ski_date"] == prev_day)
        ]

        if not prior_match.empty:
            snowfall_prev_day_cm = float(prior_match.iloc[0]["snowfall_day_total_cm"])
            rain_prev_day_mm = float(prior_match.iloc[0]["rain_day_total_mm"])
            hours_above_freezing_prev_day = int(prior_match.iloc[0]["hours_above_freezing_day"])
        else:
            snowfall_prev_day_cm = 0.0
            rain_prev_day_mm = 0.0
            hours_above_freezing_prev_day = 0

        # previous 48h freeze-thaw
        prev_48h_slice = df[
            (df["resort"] == resort) &
            (df["forecast_run_at"] == forecast_run_at) &
            (df["time_local"] >= prev_48h_start) &
            (df["time_local"] < prev_48h_end)
        ].sort_values("time_local")

        freeze_thaw_cycles_prev_48h = count_freeze_thaw_cycles(prev_48h_slice["temperature_2m_c"])

        # derived flags
        ice_risk_flag = int(
            (rain_prev_day_mm > 0 and hours_above_freezing_prev_day > 0 and freeze_thaw_cycles_prev_48h > 0)
        )

        wind_hold_risk_flag = int(
            pd.notna(row["max_wind_gust_ski_hours_kmh"]) and float(row["max_wind_gust_ski_hours_kmh"]) >= 60
        )

        results.append({
            "resort": resort,
            "forecast_run_at": forecast_run_at,
            "target_ski_date": target_ski_date,

            "avg_temp_ski_hours_c": row["avg_temp_ski_hours_c"],
            "min_temp_ski_hours_c": row["min_temp_ski_hours_c"],
            "max_temp_ski_hours_c": row["max_temp_ski_hours_c"],
            "avg_cloudcover_ski_hours_pct": row["avg_cloudcover_ski_hours_pct"],
            "avg_wind_speed_ski_hours_kmh": row["avg_wind_speed_ski_hours_kmh"],
            "max_wind_gust_ski_hours_kmh": row["max_wind_gust_ski_hours_kmh"],

            "snowfall_ski_day_cm": row["snowfall_ski_day_cm"],
            "rain_ski_day_mm": row["rain_ski_day_mm"],

            "snowfall_prev_day_cm": snowfall_prev_day_cm,
            "rain_prev_day_mm": rain_prev_day_mm,
            "hours_above_freezing_prev_day": hours_above_freezing_prev_day,
            "freeze_thaw_cycles_prev_48h": freeze_thaw_cycles_prev_48h,

            "ice_risk_flag": ice_risk_flag,
            "wind_hold_risk_flag": wind_hold_risk_flag,
        })

    return pd.DataFrame(results)

In [57]:
forecast_fp = "data/raw/pico/Pico_forecast_20260224_1610.json"

with open(forecast_fp, "r") as f:
    forecast_payload = json.load(f)

hourly_df = parse_openmeteo_hourly(
    payload=forecast_payload,
    resort="Pico",
    raw_path=forecast_fp
)

print(hourly_df.head())
print(hourly_df.shape)

features_df = build_openmeteo_daily_features(hourly_df)
print(features_df)

  resort            forecast_run_at                time_local target_ski_date  \
0   Pico  2026-02-24T21:10:00+00:00 2026-02-24 00:00:00-05:00      2026-02-24   
1   Pico  2026-02-24T21:10:00+00:00 2026-02-24 01:00:00-05:00      2026-02-24   
2   Pico  2026-02-24T21:10:00+00:00 2026-02-24 02:00:00-05:00      2026-02-24   
3   Pico  2026-02-24T21:10:00+00:00 2026-02-24 03:00:00-05:00      2026-02-24   
4   Pico  2026-02-24T21:10:00+00:00 2026-02-24 04:00:00-05:00      2026-02-24   

   hour_local  temperature_2m_c  cloudcover_pct  wind_speed_10m_kmh  \
0           0              -9.6              61                10.5   
1           1             -10.5               0                16.6   
2           2             -10.3              93                13.3   
3           3             -11.1               6                10.1   
4           4             -11.2              57                13.7   

   wind_gusts_10m_kmh  precipitation_mm  snowfall_cm  
0                32.0          

In [58]:
def extract_forecast_run_at_from_filename(fp: str | Path):
    fp = Path(fp)
    m = re.search(r"(\d{8}_\d{4})\.json$", fp.name)
    if not m:
        return None

    ts = pd.to_datetime(m.group(1), format="%Y%m%d_%H%M")
    return ts.tz_localize("America/New_York").tz_convert("UTC").isoformat()


def rain_proxy_mm(temp_c, precip_mm):
    if temp_c is None or precip_mm is None:
        return 0.0
    try:
        return float(precip_mm) if float(temp_c) > 0 and float(precip_mm) > 0 else 0.0
    except Exception:
        return 0.0


def count_freeze_thaw_cycles(series_temp_c: pd.Series) -> int:
    s = series_temp_c.dropna().reset_index(drop=True)
    if len(s) < 2:
        return 0

    prev = s.shift(1)
    crossings = ((prev <= 0) & (s > 0)) | ((prev > 0) & (s <= 0))
    return int(crossings.sum())

In [71]:
def parse_openmeteo_hourly(payload: dict, resort: str, raw_path: str) -> pd.DataFrame:
    forecast_run_at = extract_forecast_run_at_from_filename(raw_path)
    timezone_name = payload.get("timezone", "America/New_York")

    hourly = payload.get("hourly", {}) or {}

    df = pd.DataFrame({
        "time_raw": hourly.get("time", []) or [],
        "temperature_2m_c": hourly.get("temperature_2m", []) or [],
        "cloudcover_pct": hourly.get("cloudcover", []) or [],
        "wind_speed_10m_kmh": hourly.get("wind_speed_10m", []) or [],
        "wind_gusts_10m_kmh": hourly.get("wind_gusts_10m", []) or [],
        "precipitation_mm": hourly.get("precipitation", []) or [],
        "snowfall_cm": hourly.get("snowfall", []) or [],
    })

    if df.empty:
        return df

    df["time_local"] = pd.to_datetime(df["time_raw"])

    if df["time_local"].dt.tz is None:
        df["time_local"] = df["time_local"].dt.tz_localize(
            timezone_name,
            nonexistent="shift_forward",
            ambiguous="infer"
        )

    df["resort"] = resort
    df["forecast_run_at"] = forecast_run_at
    df["target_ski_date"] = df["time_local"].dt.date
    df["hour_local"] = df["time_local"].dt.hour

    return df[
        [
            "resort",
            "forecast_run_at",
            "time_local",
            "target_ski_date",
            "hour_local",
            "temperature_2m_c",
            "cloudcover_pct",
            "wind_speed_10m_kmh",
            "wind_gusts_10m_kmh",
            "precipitation_mm",
            "snowfall_cm",
        ]
    ].copy()

In [72]:
def upsert_openmeteo_hourly(df: pd.DataFrame):
    if df.empty:
        print("No hourly rows to insert.")
        return

    df = df.drop_duplicates(subset=["resort", "forecast_run_at", "time_local"])

    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        dbname="ski_pipeline",
        user="airflow",
        password="airflow",
    )
    conn.autocommit = False

    sql = """
    INSERT INTO openmeteo_hourly (
        resort,
        forecast_run_at,
        time_local,
        target_ski_date,
        hour_local,
        temperature_2m_c,
        cloudcover_pct,
        wind_speed_10m_kmh,
        wind_gusts_10m_kmh,
        precipitation_mm,
        snowfall_cm
    )
    VALUES %s
    ON CONFLICT (resort, forecast_run_at, time_local)
    DO NOTHING;
    """

    values = [
        (
            row["resort"],
            row["forecast_run_at"],
            row["time_local"].to_pydatetime() if hasattr(row["time_local"], "to_pydatetime") else row["time_local"],
            row["target_ski_date"],
            int(row["hour_local"]),
            row["temperature_2m_c"],
            row["cloudcover_pct"],
            row["wind_speed_10m_kmh"],
            row["wind_gusts_10m_kmh"],
            row["precipitation_mm"],
            row["snowfall_cm"],
        )
        for _, row in df.iterrows()
    ]

    try:
        with conn.cursor() as cur:
            execute_values(cur, sql, values, page_size=500)
        conn.commit()
        print(f"Hourly rows attempted: {len(values)}")
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

        

In [73]:
def upsert_openmeteo_features_daily(df: pd.DataFrame):
    if df.empty:
        print("No daily feature rows to insert.")
        return

    df = df.drop_duplicates(subset=["resort", "forecast_run_at", "target_ski_date"])

    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        dbname="ski_pipeline",
        user="airflow",
        password="airflow",
    )
    conn.autocommit = False

    sql = """
    INSERT INTO openmeteo_features_daily (
        resort,
        forecast_run_at,
        target_ski_date,
        avg_temp_ski_hours_c,
        min_temp_ski_hours_c,
        max_temp_ski_hours_c,
        avg_cloudcover_ski_hours_pct,
        avg_wind_speed_ski_hours_kmh,
        max_wind_gust_ski_hours_kmh,
        snowfall_ski_day_cm,
        rain_ski_day_mm,
        snowfall_prev_day_cm,
        rain_prev_day_mm,
        hours_above_freezing_prev_day,
        freeze_thaw_cycles_prev_48h,
        ice_risk_flag,
        wind_hold_risk_flag
    )
    VALUES %s
    ON CONFLICT (resort, forecast_run_at, target_ski_date)
    DO NOTHING;
    """

    values = [
        (
            row["resort"],
            row["forecast_run_at"],
            row["target_ski_date"],
            row["avg_temp_ski_hours_c"],
            row["min_temp_ski_hours_c"],
            row["max_temp_ski_hours_c"],
            row["avg_cloudcover_ski_hours_pct"],
            row["avg_wind_speed_ski_hours_kmh"],
            row["max_wind_gust_ski_hours_kmh"],
            row["snowfall_ski_day_cm"],
            row["rain_ski_day_mm"],
            row["snowfall_prev_day_cm"],
            row["rain_prev_day_mm"],
            row["hours_above_freezing_prev_day"],
            row["freeze_thaw_cycles_prev_48h"],
            row["ice_risk_flag"],
            row["wind_hold_risk_flag"],
        )
        for _, row in df.iterrows()
    ]

    try:
        with conn.cursor() as cur:
            execute_values(cur, sql, values, page_size=200)
        conn.commit()
        print(f"Daily feature rows attempted: {len(values)}")
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

In [74]:
def load_openmeteo_folder(resort: str, folder: str, pattern: str = "*_forecast_*.json"):
    folder_path = Path(folder)
    files = sorted(folder_path.glob(pattern))
    print(f"Found {len(files)} forecast files for {resort}")

    hourly_dfs = []
    feature_dfs = []

    for fp in files:
        with open(fp, "r") as f:
            payload = json.load(f)

        hourly_df = parse_openmeteo_hourly(
            payload=payload,
            resort=resort,
            raw_path=str(fp)
        )

        features_df = build_openmeteo_daily_features(hourly_df)

        hourly_dfs.append(hourly_df)
        feature_dfs.append(features_df)

    all_hourly = pd.concat(hourly_dfs, ignore_index=True) if hourly_dfs else pd.DataFrame()
    all_features = pd.concat(feature_dfs, ignore_index=True) if feature_dfs else pd.DataFrame()

    return all_hourly, all_features

In [ ]:
hourly_loon, features_loon = load_openmeteo_folder(
    resort="Loon",
    folder="data/raw/loon",
    pattern="Loon_forecast_*.json"
)

print(hourly_loon.shape)
print(features_loon.shape)

upsert_openmeteo_hourly(hourly_loon)

Found 45 forecast files for Loon
(8640, 11)
(360, 17)
Hourly rows attempted: 8633
Daily feature rows attempted: 360


In [78]:
features_loon

,resort,forecast_run_at,target_ski_date,avg_temp_ski_hours_c,min_temp_ski_hours_c,max_temp_ski_hours_c,avg_cloudcover_ski_hours_pct,avg_wind_speed_ski_hours_kmh,max_wind_gust_ski_hours_kmh,snowfall_ski_day_cm,rain_ski_day_mm,snowfall_prev_day_cm,rain_prev_day_mm,hours_above_freezing_prev_day,freeze_thaw_cycles_prev_48h,ice_risk_flag,wind_hold_risk_flag
0,Loon,2026-02-24T21:10:00+00:00,2026-02-24,-11.877778,-13.0,-11.1,25.888889,22.066667,46.8,0.00,0.0,0.00,0.0,0,0,0,0
1,Loon,2026-02-24T21:10:00+00:00,2026-02-25,-7.511111,-12.1,-4.0,100.000000,15.366667,41.4,2.10,0.0,0.00,0.0,0,0,0,0
2,Loon,2026-02-24T21:10:00+00:00,2026-02-26,-6.811111,-9.5,-5.7,76.888889,11.900000,42.1,0.00,0.0,3.92,0.0,0,0,0,0
3,Loon,2026-02-24T21:10:00+00:00,2026-02-27,-7.522222,-13.3,-5.1,19.222222,7.244444,15.1,0.00,0.0,0.00,0.0,0,0,0,0
4,Loon,2026-02-24T21:10:00+00:00,2026-02-28,-1.966667,-5.5,-0.6,100.000000,11.777778,51.8,0.49,0.0,0.00,0.0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,Loon,2026-03-23T15:00:00+00:00,2026-03-26,-0.511111,-2.0,1.5,100.000000,4.455556,15.5,1.26,0.0,1.04,0.0,4,2,0,0
356,Loon,2026-03-23T15:00:00+00:00,2026-03-27,-3.000000,-5.6,0.6,100.000000,24.555556,57.2,0.21,0.0,5.39,7.7,10,3,1,0
357,Loon,2026-03-23T15:00:00+00:00,2026-03-28,-10.655556,-16.1,-6.4,88.777778,12.233333,34.6,0.00,0.0,0.28,10.2,9,2,1,0
358,Loon,2026-03-23T15:00:00+00:00,2026-03-29,-3.855556,-7.1,-0.8,58.555556,13.222222,32.0,0.00,0.0,0.21,0.0,0,1,0,0


In [77]:

upsert_openmeteo_features_daily(features_loon)

Daily feature rows attempted: 360


In [79]:
openmeteo_jobs = [
    ("Loon", "data/raw/loon", "Loon_forecast_*.json"),
    ("SundayRiver", "data/raw/sundayriver", "SundayRiver_forecast_*.json"),
    ("Sugarloaf", "data/raw/sugarloaf", "Sugarloaf_forecast_*.json"),
    ("Stratton", "data/raw/stratton", "Stratton_forecast_*.json"),
    ("Sugarbush", "data/raw/sugarbush", "Sugarbush_forecast_*.json"),
    ("Killington", "data/raw/killington", "Killington_forecast_*.json"),
    ("Pico", "data/raw/pico", "Pico_forecast_*.json"),
]

all_hourly = []
all_features = []

for resort, folder, pattern in openmeteo_jobs:
    print(f"\n=== Loading {resort} ===")

    hourly_df, features_df = load_openmeteo_folder(
        resort=resort,
        folder=folder,
        pattern=pattern
    )

    print(f"{resort} hourly shape: {hourly_df.shape}")
    print(f"{resort} feature shape: {features_df.shape}")

    upsert_openmeteo_hourly(hourly_df)
    upsert_openmeteo_features_daily(features_df)

    all_hourly.append(hourly_df)
    all_features.append(features_df)


=== Loading Loon ===
Found 45 forecast files for Loon
Loon hourly shape: (8640, 11)
Loon feature shape: (360, 17)
Hourly rows attempted: 8633
Daily feature rows attempted: 360

=== Loading SundayRiver ===
Found 45 forecast files for SundayRiver
SundayRiver hourly shape: (8640, 11)
SundayRiver feature shape: (360, 17)
Hourly rows attempted: 8633
Daily feature rows attempted: 360

=== Loading Sugarloaf ===
Found 45 forecast files for Sugarloaf
Sugarloaf hourly shape: (8640, 11)
Sugarloaf feature shape: (360, 17)
Hourly rows attempted: 8633
Daily feature rows attempted: 360

=== Loading Stratton ===
Found 45 forecast files for Stratton
Stratton hourly shape: (8640, 11)
Stratton feature shape: (360, 17)
Hourly rows attempted: 8633
Daily feature rows attempted: 360

=== Loading Sugarbush ===
Found 45 forecast files for Sugarbush
Sugarbush hourly shape: (8640, 11)
Sugarbush feature shape: (360, 17)
Hourly rows attempted: 8633
Daily feature rows attempted: 360

=== Loading Killington ===
Fou